## Setup and Imports

In [ ]:
# Standard library imports
import sys
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import numpy as np

# Data manipulation
import pandas as pd
import seaborn as sns

# Statistical analysis
from scipy import stats
from scipy.signal import find_peaks

# Configuration
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# Add project modules to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import custom modules

print("Libraries imported successfully")
print(f"Project root: {project_root}")

## 1. Rolling Window Configuration

In [ ]:
# Configuration
DATA_DIR = project_root / "data" / "output"  # Preprocessed data location
RAW_DATA_PATH = project_root / "data" / "input" / "train" / "azure_ad_train.jsonl"  # Raw training data

# Rolling window parameters
WINDOW_SIZE = "1H"  # 1 hour window
MIN_PERIODS = 1  # Minimum observations per window
AGGREGATION_FUNCTIONS = ["sum", "mean", "std", "min", "max", "count"]

print("Rolling Window Configuration:")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Min periods: {MIN_PERIODS}")
print(f"  Aggregation functions: {', '.join(AGGREGATION_FUNCTIONS)}")

In [ ]:
# Load preprocessed data (with rolling window features)
if DATA_DIR.exists():
    preprocessed_files = list(DATA_DIR.glob("*_train.parquet"))

    if preprocessed_files:
        print(f"\nFound {len(preprocessed_files)} preprocessed training files")

        # Load first user for demonstration
        sample_file = preprocessed_files[0]
        username = sample_file.stem.replace("_train", "")

        print(f"\nLoading sample user: {username}")
        df_windowed = pd.read_parquet(sample_file)

        print(f"Samples: {len(df_windowed)}")
        print(f"Features: {len(df_windowed.columns)}")
        print(f"\nSample columns: {df_windowed.columns.tolist()[:20]}")
    else:
        print("No preprocessed files found.")
        print("\nPlease run preprocessing first:")
        print("python dfp-poc/pipelines/run_training_cli.py --config dfp-poc/config/pipeline.yaml")
        df_windowed = None
else:
    print(f"Data directory not found: {DATA_DIR}")
    df_windowed = None

In [ ]:
# Alternatively, load raw data and create rolling windows manually
if df_windowed is None and RAW_DATA_PATH.exists():
    print("Loading raw data for rolling window demonstration...")

    df_raw = pd.read_json(RAW_DATA_PATH, lines=True)
    df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])

    # Select a sample user with sufficient data
    user_counts = df_raw["username"].value_counts()
    sample_username = user_counts[user_counts > 100].index[0]

    df_user = df_raw[df_raw["username"] == sample_username].copy()
    df_user = df_user.sort_values("timestamp")

    print(f"\nSample user: {sample_username}")
    print(f"Events: {len(df_user)}")
    print(f"Date range: {df_user['timestamp'].min()} to {df_user['timestamp'].max()}")

    # Simple rolling window example on numeric features
    numeric_features = df_user.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_features:
        df_user = df_user.set_index("timestamp")

        # Create rolling window features
        df_windowed_manual = pd.DataFrame(index=df_user.index)

        for feature in numeric_features[:5]:  # Limit to first 5 features
            rolling = df_user[feature].rolling(window=WINDOW_SIZE, min_periods=MIN_PERIODS)

            df_windowed_manual[f"{feature}_sum"] = rolling.sum()
            df_windowed_manual[f"{feature}_mean"] = rolling.mean()
            df_windowed_manual[f"{feature}_std"] = rolling.std()

        print(f"\nCreated rolling window features: {len(df_windowed_manual.columns)}")
        df_windowed = df_windowed_manual
        username = sample_username
    else:
        print("No numeric features found for rolling window")
        df_windowed = None
elif not RAW_DATA_PATH.exists():
    print(f"Raw data not found: {RAW_DATA_PATH}")

## 2. Window Aggregation Statistics

In [ ]:
# Analyze aggregated features
if df_windowed is not None:
    # Identify aggregation types
    agg_types = {}
    for col in df_windowed.columns:
        for agg_func in ["sum", "mean", "std", "min", "max", "count"]:
            if col.endswith(f"_{agg_func}"):
                agg_types.setdefault(agg_func, []).append(col)
                break

    print("Aggregation Feature Breakdown:")
    for agg_func, cols in agg_types.items():
        print(f"  {agg_func}: {len(cols)} features")

    # Statistical summary
    print("\nRolling Window Feature Statistics:")
    print(df_windowed.describe())

In [ ]:
# Compare aggregation statistics
if df_windowed is not None and agg_types:
    # Select features with 'mean' aggregation
    mean_features = [col for col in df_windowed.columns if col.endswith("_mean")][:10]

    if mean_features:
        fig, axes = plt.subplots(2, 1, figsize=(15, 10))

        # Box plots
        df_windowed[mean_features].boxplot(ax=axes[0], vert=False)
        axes[0].set_xlabel("Value", fontsize=12)
        axes[0].set_title("Rolling Window Mean Features Distribution", fontsize=14, fontweight="bold")
        axes[0].grid(True, alpha=0.3)

        # Histograms for top 4 features
        for idx, feature in enumerate(mean_features[:4]):
            if idx == 0:
                ax = axes[1]
            else:
                ax = axes[1]

            axes[1].hist(df_windowed[feature].dropna(), bins=30, alpha=0.5, label=feature, edgecolor="black")

        axes[1].set_xlabel("Value", fontsize=12)
        axes[1].set_ylabel("Frequency", fontsize=12)
        axes[1].set_title("Sample Rolling Window Feature Distributions", fontsize=14, fontweight="bold")
        axes[1].legend(fontsize=9, loc="upper right")
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

## 3. Per-User Time Series Analysis

In [ ]:
# Time series plots for key behavioral features
if df_windowed is not None:
    # Look for behavioral features (logcount, locincrement, appincrement)
    behavioral_features = []
    for prefix in ["logcount", "locincrement", "appincrement"]:
        matching = [col for col in df_windowed.columns if col.startswith(prefix)]
        behavioral_features.extend(matching[:2])  # Take first 2 aggregations of each

    if not behavioral_features:
        # Fallback to any numeric features
        behavioral_features = df_windowed.select_dtypes(include=[np.number]).columns.tolist()[:6]

    if behavioral_features and isinstance(df_windowed.index, pd.DatetimeIndex):
        fig, axes = plt.subplots(len(behavioral_features[:6]), 1, figsize=(16, 3 * len(behavioral_features[:6])))

        if len(behavioral_features[:6]) == 1:
            axes = [axes]

        for idx, feature in enumerate(behavioral_features[:6]):
            axes[idx].plot(df_windowed.index, df_windowed[feature], linewidth=1.5, marker="o", markersize=3)
            axes[idx].set_ylabel(feature, fontsize=11)
            axes[idx].set_title(f"Time Series: {feature}", fontsize=12, fontweight="bold")
            axes[idx].grid(True, alpha=0.3)
            axes[idx].tick_params(axis="x", rotation=45)

        axes[-1].set_xlabel("Timestamp", fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("No suitable features or time index for time series visualization")

In [ ]:
# Detect patterns and anomalies in time series
if df_windowed is not None and behavioral_features:
    # Select first behavioral feature for detailed analysis
    feature = behavioral_features[0]
    series = df_windowed[feature].dropna()

    if len(series) > 10:
        # Detect peaks (spikes in activity)
        peaks, properties = find_peaks(series, prominence=series.std())

        # Detect valleys (drops in activity)
        valleys, _ = find_peaks(-series, prominence=series.std())

        plt.figure(figsize=(16, 6))
        plt.plot(series.index, series.values, linewidth=2, label="Feature Value")

        if len(peaks) > 0:
            plt.scatter(
                series.index[peaks],
                series.values[peaks],
                color="red",
                s=100,
                marker="^",
                label=f"Peaks ({len(peaks)})",
                zorder=5,
            )

        if len(valleys) > 0:
            plt.scatter(
                series.index[valleys],
                series.values[valleys],
                color="blue",
                s=100,
                marker="v",
                label=f"Valleys ({len(valleys)})",
                zorder=5,
            )

        # Add mean and threshold lines
        plt.axhline(series.mean(), color="green", linestyle="--", linewidth=1.5, label="Mean")
        plt.axhline(
            series.mean() + 2 * series.std(), color="orange", linestyle="--", linewidth=1.5, label="Mean + 2 Std"
        )

        plt.xlabel("Timestamp", fontsize=12)
        plt.ylabel(feature, fontsize=12)
        plt.title(f"Pattern Detection: {feature} for {username}", fontsize=14, fontweight="bold")
        plt.legend(fontsize=10, loc="upper right")
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print(f"\nPattern Analysis for {feature}:")
        print(f"  Peaks detected: {len(peaks)}")
        print(f"  Valleys detected: {len(valleys)}")
        print(f"  Values above 2-sigma: {np.sum(series > series.mean() + 2 * series.std())}")

## 4. Feature Evolution Over Time

In [ ]:
# Compare sum vs mean vs std aggregations for the same base feature
if df_windowed is not None:
    # Find base features with multiple aggregations
    base_features = set()
    for col in df_windowed.columns:
        for agg in ["sum", "mean", "std"]:
            if col.endswith(f"_{agg}"):
                base = col.replace(f"_{agg}", "")
                base_features.add(base)

    # Select first base feature
    if base_features:
        base_feature = list(base_features)[0]
        agg_columns = [f"{base_feature}_sum", f"{base_feature}_mean", f"{base_feature}_std"]
        available_aggs = [col for col in agg_columns if col in df_windowed.columns]

        if len(available_aggs) >= 2 and isinstance(df_windowed.index, pd.DatetimeIndex):
            fig, axes = plt.subplots(len(available_aggs), 1, figsize=(16, 4 * len(available_aggs)))

            if len(available_aggs) == 1:
                axes = [axes]

            for idx, agg_col in enumerate(available_aggs):
                axes[idx].plot(
                    df_windowed.index, df_windowed[agg_col], linewidth=2, marker="o", markersize=4, color=f"C{idx}"
                )
                axes[idx].set_ylabel(agg_col, fontsize=12)
                axes[idx].set_title(f"{agg_col} Over Time", fontsize=13, fontweight="bold")
                axes[idx].grid(True, alpha=0.3)
                axes[idx].tick_params(axis="x", rotation=45)

            axes[-1].set_xlabel("Timestamp", fontsize=12)
            plt.suptitle(
                f"Feature Evolution: {base_feature} (Multiple Aggregations)", fontsize=15, fontweight="bold", y=1.001
            )
            plt.tight_layout()
            plt.show()

## 5. Window Size Impact Analysis

In [ ]:
# Compare different window sizes (requires raw data)
if RAW_DATA_PATH.exists():
    print("Analyzing impact of different window sizes...")

    # Load raw data
    df_raw = pd.read_json(RAW_DATA_PATH, lines=True)
    df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])

    # Select user
    user_counts = df_raw["username"].value_counts()
    sample_user = user_counts[user_counts > 100].index[0]
    df_user_raw = df_raw[df_raw["username"] == sample_user].copy().sort_values("timestamp")
    df_user_raw = df_user_raw.set_index("timestamp")

    # Get a numeric column for demonstration
    numeric_cols = df_user_raw.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        demo_column = numeric_cols[0]

        # Test different window sizes
        window_sizes = ["15T", "30T", "1H", "2H", "6H"]  # minutes/hours

        fig, axes = plt.subplots(len(window_sizes), 1, figsize=(16, 4 * len(window_sizes)))

        for idx, window_size in enumerate(window_sizes):
            rolling = df_user_raw[demo_column].rolling(window=window_size, min_periods=1)
            rolling_mean = rolling.mean()

            axes[idx].plot(df_user_raw.index, df_user_raw[demo_column], alpha=0.3, label="Original", linewidth=1)
            axes[idx].plot(
                rolling_mean.index,
                rolling_mean.values,
                label=f"Rolling Mean ({window_size})",
                linewidth=2.5,
                color="red",
            )
            axes[idx].set_ylabel(demo_column, fontsize=11)
            axes[idx].set_title(f"Window Size: {window_size}", fontsize=12, fontweight="bold")
            axes[idx].legend(fontsize=10)
            axes[idx].grid(True, alpha=0.3)
            axes[idx].tick_params(axis="x", rotation=45)

        axes[-1].set_xlabel("Timestamp", fontsize=12)
        plt.suptitle(
            f"Impact of Window Size on Feature Smoothing: {demo_column}", fontsize=15, fontweight="bold", y=1.001
        )
        plt.tight_layout()
        plt.show()

        print("\nObservations:")
        print("  - Smaller windows (15T, 30T) preserve more detail but are noisier")
        print("  - Larger windows (2H, 6H) smooth out noise but may miss short-term anomalies")
        print("  - 1H window provides good balance for hourly behavior patterns")
    else:
        print("No numeric columns available for window size analysis")
else:
    print(f"Raw data not available: {RAW_DATA_PATH}")

## 6. Correlation Dynamics

In [ ]:
# Analyze correlation between windowed features
if df_windowed is not None:
    # Select numeric columns
    numeric_windowed = df_windowed.select_dtypes(include=[np.number])

    if len(numeric_windowed.columns) > 1:
        # Compute correlation matrix
        corr_matrix = numeric_windowed.corr()

        # Select subset for visualization (first 20 features)
        subset_size = min(20, len(corr_matrix))
        corr_subset = corr_matrix.iloc[:subset_size, :subset_size]

        plt.figure(figsize=(14, 12))
        sns.heatmap(
            corr_subset, annot=False, cmap="coolwarm", center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}
        )
        plt.title("Rolling Window Feature Correlations", fontsize=14, fontweight="bold", pad=15)
        plt.tight_layout()
        plt.show()

        # Print highly correlated pairs
        print("\nHighly Correlated Feature Pairs (|correlation| > 0.8):")
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i + 1, len(corr_matrix.columns)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.8:
                    col1 = corr_matrix.columns[i]
                    col2 = corr_matrix.columns[j]
                    high_corr_pairs.append((col1, col2, corr_val))

        for col1, col2, corr_val in high_corr_pairs[:15]:  # Show first 15
            print(f"  {col1} <-> {col2}: {corr_val:.3f}")

        if len(high_corr_pairs) > 15:
            print(f"  ... and {len(high_corr_pairs) - 15} more pairs")

In [ ]:
# Scatter plots for highly correlated features
if df_windowed is not None and len(high_corr_pairs) > 0:
    # Select top 4 pairs for visualization
    top_pairs = sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:4]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()

    for idx, (col1, col2, corr_val) in enumerate(top_pairs):
        axes[idx].scatter(df_windowed[col1], df_windowed[col2], alpha=0.5, s=30)
        axes[idx].set_xlabel(col1, fontsize=10)
        axes[idx].set_ylabel(col2, fontsize=10)
        axes[idx].set_title(f"Correlation: {corr_val:.3f}", fontsize=11, fontweight="bold")
        axes[idx].grid(True, alpha=0.3)

        # Add regression line
        valid_data = df_windowed[[col1, col2]].dropna()
        if len(valid_data) > 1:
            z = np.polyfit(valid_data[col1], valid_data[col2], 1)
            p = np.poly1d(z)
            axes[idx].plot(valid_data[col1], p(valid_data[col1]), "r--", linewidth=2, alpha=0.8)

    plt.suptitle("Highly Correlated Rolling Window Features", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 7. Anomaly Pattern Detection

In [ ]:
# Detect anomalies using statistical methods on windowed features
if df_windowed is not None and behavioral_features:
    # Select feature for anomaly detection
    anomaly_feature = behavioral_features[0]
    feature_data = df_windowed[anomaly_feature].dropna()

    if len(feature_data) > 10:
        # Method 1: Z-score
        z_scores = np.abs(stats.zscore(feature_data))
        z_threshold = 3
        z_anomalies = z_scores > z_threshold

        # Method 2: IQR
        Q1 = feature_data.quantile(0.25)
        Q3 = feature_data.quantile(0.75)
        IQR = Q3 - Q1
        iqr_lower = Q1 - 1.5 * IQR
        iqr_upper = Q3 + 1.5 * IQR
        iqr_anomalies = (feature_data < iqr_lower) | (feature_data > iqr_upper)

        # Method 3: Rolling statistics
        rolling_mean = feature_data.rolling(window=10, min_periods=1).mean()
        rolling_std = feature_data.rolling(window=10, min_periods=1).std()
        rolling_anomalies = np.abs(feature_data - rolling_mean) > (2 * rolling_std)

        print(f"Anomaly Detection Results for {anomaly_feature}:")
        print(
            f"  Z-score method (|z| > {z_threshold}): {z_anomalies.sum()} anomalies ({(z_anomalies.sum() / len(feature_data)) * 100:.2f}%)"
        )
        print(f"  IQR method: {iqr_anomalies.sum()} anomalies ({(iqr_anomalies.sum() / len(feature_data)) * 100:.2f}%)")
        print(
            f"  Rolling statistics: {rolling_anomalies.sum()} anomalies ({(rolling_anomalies.sum() / len(feature_data)) * 100:.2f}%)"
        )

        # Visualize anomalies
        fig, axes = plt.subplots(3, 1, figsize=(16, 12))

        # Z-score method
        axes[0].plot(feature_data.index, feature_data.values, linewidth=1.5, label="Feature Value")
        axes[0].scatter(
            feature_data.index[z_anomalies],
            feature_data.values[z_anomalies],
            color="red",
            s=100,
            marker="x",
            label=f"Anomalies (n={z_anomalies.sum()})",
            zorder=5,
        )
        axes[0].set_ylabel(anomaly_feature, fontsize=11)
        axes[0].set_title("Anomaly Detection: Z-Score Method", fontsize=13, fontweight="bold")
        axes[0].legend(fontsize=10)
        axes[0].grid(True, alpha=0.3)

        # IQR method
        axes[1].plot(feature_data.index, feature_data.values, linewidth=1.5, label="Feature Value")
        axes[1].scatter(
            feature_data.index[iqr_anomalies],
            feature_data.values[iqr_anomalies],
            color="orange",
            s=100,
            marker="x",
            label=f"Anomalies (n={iqr_anomalies.sum()})",
            zorder=5,
        )
        axes[1].axhline(iqr_lower, color="green", linestyle="--", linewidth=1.5, label="IQR Bounds")
        axes[1].axhline(iqr_upper, color="green", linestyle="--", linewidth=1.5)
        axes[1].set_ylabel(anomaly_feature, fontsize=11)
        axes[1].set_title("Anomaly Detection: IQR Method", fontsize=13, fontweight="bold")
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)

        # Rolling statistics method
        axes[2].plot(feature_data.index, feature_data.values, linewidth=1.5, label="Feature Value")
        axes[2].scatter(
            feature_data.index[rolling_anomalies],
            feature_data.values[rolling_anomalies],
            color="purple",
            s=100,
            marker="x",
            label=f"Anomalies (n={rolling_anomalies.sum()})",
            zorder=5,
        )
        axes[2].plot(
            rolling_mean.index, rolling_mean.values, color="green", linestyle="--", linewidth=1.5, label="Rolling Mean"
        )
        axes[2].fill_between(
            rolling_mean.index,
            rolling_mean - 2 * rolling_std,
            rolling_mean + 2 * rolling_std,
            alpha=0.2,
            color="green",
            label="2-Sigma Band",
        )
        axes[2].set_ylabel(anomaly_feature, fontsize=11)
        axes[2].set_xlabel("Timestamp", fontsize=12)
        axes[2].set_title("Anomaly Detection: Rolling Statistics Method", fontsize=13, fontweight="bold")
        axes[2].legend(fontsize=10)
        axes[2].grid(True, alpha=0.3)

        for ax in axes:
            ax.tick_params(axis="x", rotation=45)

        plt.suptitle(f"Anomaly Detection Comparison for {username}", fontsize=15, fontweight="bold", y=1.001)
        plt.tight_layout()
        plt.show()

## Summary and Recommendations

In [ ]:
print("=" * 80)
print("ROLLING WINDOW VISUALIZATION SUMMARY")
print("=" * 80)

if df_windowed is not None:
    print("\n1. ROLLING WINDOW CONFIGURATION")
    print(f"   Window size: {WINDOW_SIZE}")
    print(f"   Total windowed features: {len(df_windowed.columns)}")
    print(f"   Samples analyzed: {len(df_windowed)}")

    if agg_types:
        print("\n2. AGGREGATION BREAKDOWN")
        for agg_func, cols in agg_types.items():
            print(f"   {agg_func}: {len(cols)} features")

if behavioral_features:
    print("\n3. TEMPORAL PATTERNS")
    print(f"   Key behavioral features identified: {len(behavioral_features)}")
    if "peaks" in locals():
        print(f"   Activity peaks detected: {len(peaks)}")
        print(f"   Activity valleys detected: {len(valleys)}")

if "high_corr_pairs" in locals():
    print("\n4. FEATURE CORRELATIONS")
    print(f"   Highly correlated pairs (|r| > 0.8): {len(high_corr_pairs)}")
    if high_corr_pairs:
        print(f"   Strongest correlation: {max(high_corr_pairs, key=lambda x: abs(x[2]))[2]:.3f}")

if "z_anomalies" in locals():
    print("\n5. ANOMALY DETECTION")
    print(f"   Z-score anomalies: {z_anomalies.sum()}")
    print(f"   IQR anomalies: {iqr_anomalies.sum()}")
    print(f"   Rolling statistics anomalies: {rolling_anomalies.sum()}")

print("\n6. RECOMMENDATIONS")
print("   - 1-hour rolling window provides good balance for behavioral analysis")
print("   - Multiple aggregation functions (sum, mean, std) capture different aspects")
print("   - Temporal patterns reveal normal behavior baselines")
print("   - Statistical anomaly detection methods show consistency")
print("   - Rolling window preprocessing reduces noise while preserving trends")

print("\n" + "=" * 80)